In [29]:
# Импорт и чтение файла

In [30]:
import pandas as pd
import numpy as np

file_path = "Chapter3_dataset.xlsx"

# читаем первый лист
df_raw = pd.read_excel(file_path, sheet_name="1")

df_raw.head()

,Patients-GeneSymbol,ELMO2,CREB3L1,RPS11,PNMA1,MMP2,C10orf90,ZHX3,Unnamed: 8,Unnamed: 9,Unnamed: 10,C1,C2,C3,C4
0,P1,-0.840833,1.33300,1.475500,-1.39075,0.027833,-2.54950,-1.242000,NaN,NaN,ELMO2,-5.0,1.0,2.0,2.0
1,P2,0.018000,0.76250,0.324500,-1.48725,-0.206500,-2.76575,-0.303000,NaN,NaN,CREB3L1,3.0,-3.0,-3.0,1.0
2,P3,0.354917,0.13475,0.634000,-0.83050,0.066667,-2.76375,-0.013167,NaN,NaN,RPS11,1.0,3.0,3.0,-1.0
3,P4,-0.203750,2.49875,0.483125,-0.46325,2.156000,-2.48225,-0.063500,NaN,NaN,PNMA1,-2.0,-2.0,2.0,-4.0
4,P5,0.689000,1.15725,0.952625,-2.22300,0.150167,-2.11075,-0.609000,NaN,NaN,MMP2,0.0,-3.0,-4.0,-1.0


In [31]:
# Отделяем данные пациентов

In [32]:
# признаки генов
gene_columns = [
    "ELMO2",
    "CREB3L1",
    "RPS11",
    "PNMA1",
    "MMP2",
    "C10orf90",
    "ZHX3"
]

patients = df_raw["Patients-GeneSymbol"]
X = df_raw[gene_columns].copy()

X.head()

,ELMO2,CREB3L1,RPS11,PNMA1,MMP2,C10orf90,ZHX3
0,-0.840833,1.33300,1.475500,-1.39075,0.027833,-2.54950,-1.242000
1,0.018000,0.76250,0.324500,-1.48725,-0.206500,-2.76575,-0.303000
2,0.354917,0.13475,0.634000,-0.83050,0.066667,-2.76375,-0.013167
3,-0.203750,2.49875,0.483125,-0.46325,2.156000,-2.48225,-0.063500
4,0.689000,1.15725,0.952625,-2.22300,0.150167,-2.11075,-0.609000


In [33]:
# Начальные центроиды

In [34]:
initial_centroids = pd.DataFrame(
    {
        "C1": [-5, 3, 1, -2, 0, -4, 3],
        "C2": [1, -3, 3, -2, -3, 1, 2],
        "C3": [2, -3, 3, 2, -4, -1, -3],
        "C4": [2, 1, -1, -4, -1, -5, 2],
    },
    index=gene_columns
)

initial_centroids

,C1,C2,C3,C4
ELMO2,-5,1,2,2
CREB3L1,3,-3,-3,1
RPS11,1,3,3,-1
PNMA1,-2,-2,2,-4
MMP2,0,-3,-4,-1
C10orf90,-4,1,-1,-5
ZHX3,3,2,-3,2


In [35]:
# Функции K-Means
# Алгоритм делает 4 вещи:

# считает расстояния до центров кластеров,
# выбирает ближайший кластер,
# пересчитывает центры,
# повторяет это, пока ничего не меняется.

In [36]:
# Считает расстояние от каждого пациента до каждого центра кластера.
def compute_distances(X, centroids):
    distances = pd.DataFrame(index=X.index)

    for cluster_name in centroids.columns:
        centroid_values = centroids[cluster_name].values
        # sqrt (x1−c1)^2+(x2−c2)^2+...+(xn−cn)^2
        distances[cluster_name] = np.sqrt(((X.values - centroid_values) ** 2).sum(axis=1))

    return distances

# Выбирает минимальное расстояние.
def assign_clusters(distances):
    # “Найди название столбца, где минимальное значение”
    return distances.idxmin(axis=1)

# Пересчитывает центр каждого кластера.
def update_centroids(X, clusters, cluster_names):
    new_centroids = pd.DataFrame(index=X.columns)

    for cluster_name in cluster_names:
        cluster_points = X[clusters == cluster_name]

        if len(cluster_points) == 0:
            new_centroids[cluster_name] = 0
        else:
            new_centroids[cluster_name] = cluster_points.mean(axis=0) # считает среднее по колонкам.

    return new_centroids

# сколько пациентов поменяли кластер.
def count_differences(old_clusters, new_clusters):
    if old_clusters is None:
        return len(new_clusters)

    return (old_clusters != new_clusters).sum()

In [37]:
# Запуск алгоритма

In [38]:
centroids = initial_centroids.copy()
old_clusters = None

history = []

for iteration in range(1, 100):
    distances = compute_distances(X, centroids) # Считаем расстояния
    clusters = assign_clusters(distances) # Назначаем кластеры
    difference = count_differences(old_clusters, clusters) # Считаем изменения

    history.append({
        "iteration": iteration,
        "difference": difference,
        "centroids": centroids.copy(),
        "clusters": clusters.copy(),
        "distances": distances.copy()
    })

    print(f"Iteration {iteration}: difference = {difference}")

    if difference == 0: # Если изменений нет: Алгоритм останавливается.
        break
        
    # Пересчитываются новые центры.
    centroids = update_centroids(X, clusters, initial_centroids.columns)
    old_clusters = clusters.copy()

Iteration 1: difference = 155
Iteration 2: difference = 46
Iteration 3: difference = 18
Iteration 4: difference = 12
Iteration 5: difference = 7
Iteration 6: difference = 7
Iteration 7: difference = 7
Iteration 8: difference = 8
Iteration 9: difference = 6
Iteration 10: difference = 4
Iteration 11: difference = 8
Iteration 12: difference = 5
Iteration 13: difference = 2
Iteration 14: difference = 2
Iteration 15: difference = 0


In [39]:
# Финальные кластеры пациентов

In [40]:
result = pd.DataFrame({
    "Patient": patients,
    "Cluster": clusters
})

result.head(20)

,Patient,Cluster
0,P1,C4
1,P2,C4
2,P3,C4
3,P4,C3
4,P5,C4
5,P6,C2
6,P7,C3
7,P8,C4
8,P9,C2
9,P10,C1


In [41]:
# Сколько пациентов в каждом кластере

In [42]:
cluster_counts = result["Cluster"].value_counts().sort_index()
cluster_counts

Cluster
C1    24
C2    49
C3    29
C4    53
Name: count, dtype: int64

In [43]:
# Финальные центроиды

In [44]:
final_centroids = history[-1]["centroids"]
final_centroids

,C1,C2,C3,C4
ELMO2,-0.566444,-0.487260,-0.495408,-0.177229
CREB3L1,2.089344,0.806566,1.652095,1.131920
RPS11,0.879120,1.116194,0.700858,0.878250
PNMA1,-1.279958,-2.150000,-0.213112,-1.478208
MMP2,-0.950382,-1.048007,0.268868,-0.039626
C10orf90,-2.126031,-2.101369,-2.386810,-2.187392
ZHX3,-0.876646,-0.442088,-0.827839,-0.405327


In [45]:
# Сохранение результата в Excel

In [46]:
output_file = "Chapter3_HW_KMeans_Result.xlsx"

with pd.ExcelWriter(output_file) as writer:
    result.to_excel(writer, sheet_name="Final clusters", index=False)
    cluster_counts.to_excel(writer, sheet_name="Cluster counts")
    final_centroids.to_excel(writer, sheet_name="Final centroids")

print(f"Saved to {output_file}")

Saved to Chapter3_HW_KMeans_Result.xlsx
